# SGD, Momentum, and Adam: Optimisers from First Principles

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/optimisers_from_scratch.ipynb)

Implement SGD, momentum, RMSProp, and Adam from scratch in NumPy. Visualise their trajectories on the same loss surface, then benchmark them training a real CNN on MNIST.

**Blog post:** [sesen.ai/blog/optimisers-sgd-momentum-adam-from-scratch](https://sesen.ai/blog/optimisers-sgd-momentum-adam-from-scratch)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

## Optimiser Implementations

We define the Rosenbrock function (narrow curved valley, minimum at (1, 1)) and four optimisers.

In [ ]:
def rosenbrock(x, y):
    """Rosenbrock function: narrow curved valley, minimum at (1, 1)."""
    return (1 - x)**2 + 100 * (y - x**2)**2

def rosenbrock_grad(x, y):
    """Gradient of the Rosenbrock function."""
    dx = -2 * (1 - x) - 400 * x * (y - x**2)
    dy = 200 * (y - x**2)
    return np.array([dx, dy])

# --- SGD ---
def sgd(grad_fn, x0, lr=0.0002, steps=2000):
    path = [x0.copy()]
    x = x0.copy()
    for _ in range(steps):
        g = grad_fn(*x)
        x -= lr * g
        path.append(x.copy())
    return np.array(path)

# --- SGD + Momentum ---
def sgd_momentum(grad_fn, x0, lr=0.0002, mom=0.9, steps=2000):
    path = [x0.copy()]
    x = x0.copy()
    v = np.zeros_like(x)
    for _ in range(steps):
        g = grad_fn(*x)
        v = mom * v + g
        x -= lr * v
        path.append(x.copy())
    return np.array(path)

# --- RMSProp ---
def rmsprop(grad_fn, x0, lr=0.001, beta=0.99, eps=1e-8, steps=2000):
    path = [x0.copy()]
    x = x0.copy()
    v = np.zeros_like(x)
    for _ in range(steps):
        g = grad_fn(*x)
        v = beta * v + (1 - beta) * g**2
        x -= lr * g / (np.sqrt(v) + eps)
        path.append(x.copy())
    return np.array(path)

# --- Adam ---
def adam(grad_fn, x0, lr=0.005, beta1=0.9, beta2=0.999, eps=1e-8, steps=2000):
    path = [x0.copy()]
    x = x0.copy()
    m = np.zeros_like(x)  # first moment (mean of gradients)
    v = np.zeros_like(x)  # second moment (mean of squared gradients)
    for t in range(1, steps + 1):
        g = grad_fn(*x)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g**2
        m_hat = m / (1 - beta1**t)  # bias correction
        v_hat = v / (1 - beta2**t)  # bias correction
        x -= lr * m_hat / (np.sqrt(v_hat) + eps)
        path.append(x.copy())
    return np.array(path)

In [ ]:
# Run all four from the same starting point
x0 = np.array([-1.0, 1.0])
path_sgd = sgd(rosenbrock_grad, x0, lr=0.0002, steps=2000)
path_mom = sgd_momentum(rosenbrock_grad, x0, lr=0.0002, mom=0.9, steps=2000)
path_rms = rmsprop(rosenbrock_grad, x0, lr=0.001, steps=2000)
path_adam = adam(rosenbrock_grad, x0, lr=0.005, steps=2000)

# Global minimum is at (1, 1)
for name, path in [("SGD", path_sgd), ("Momentum", path_mom),
                    ("RMSProp", path_rms), ("Adam", path_adam)]:
    dist = np.sqrt((path[-1][0] - 1)**2 + (path[-1][1] - 1)**2)
    print(f"{name:10s} final=({path[-1][0]:.4f}, {path[-1][1]:.4f}), dist to min={dist:.4f}")

## Trajectory Visualisation

All four optimisers on the Rosenbrock loss surface.

In [ ]:
# Plot trajectories on Rosenbrock contours
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

xx = np.linspace(-1.5, 1.5, 300)
yy = np.linspace(-0.5, 2.0, 300)
X, Y = np.meshgrid(xx, yy)
Z = rosenbrock(X, Y)

paths = [("SGD", path_sgd, '#ef4444'),
         ("Momentum", path_mom, '#3b82f6'),
         ("RMSProp", path_rms, '#22c55e'),
         ("Adam", path_adam, '#f59e0b')]

for ax, (name, path, color) in zip(axes, paths):
    ax.contour(X, Y, Z, levels=np.logspace(0, 3.5, 20), cmap='gray', alpha=0.4)
    ax.plot(path[:, 0], path[:, 1], color=color, linewidth=1.5, alpha=0.8)
    ax.plot(path[0, 0], path[0, 1], 'ko', markersize=6)
    ax.plot(1, 1, 'r*', markersize=15)
    ax.set_title(f'{name}\nfinal dist={np.sqrt((path[-1][0]-1)**2+(path[-1][1]-1)**2):.4f}',
                fontsize=11)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-0.5, 2.0)
    ax.set_aspect('equal')

plt.suptitle('Optimiser Trajectories on Rosenbrock Function (2000 steps)', fontsize=14)
plt.tight_layout()
plt.show()

## SGD vs Momentum

SGD oscillates across the valley walls while making slow progress along the floor. Momentum smooths these oscillations.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, path, color) in zip([ax1, ax2],
    [("SGD", path_sgd, '#ef4444'), ("SGD + Momentum", path_mom, '#3b82f6')]):
    ax.contour(X, Y, Z, levels=np.logspace(0, 3.5, 20), cmap='gray', alpha=0.4)
    ax.plot(path[:, 0], path[:, 1], color=color, linewidth=1.5, alpha=0.8)
    ax.plot(path[0, 0], path[0, 1], 'ko', markersize=8, label='Start')
    ax.plot(1, 1, 'r*', markersize=15, label='Minimum')
    ax.set_title(name, fontsize=13)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-0.5, 2.0)
    ax.set_aspect('equal')
    ax.legend()

plt.tight_layout()
plt.show()

## Step Functions

Compact implementations showing each optimiser's core update rule.

In [ ]:
def sgd_step(params, grads, lr):
    """One step of vanilla SGD. This is the entire algorithm."""
    for p, g in zip(params, grads):
        p -= lr * g

def sgd_momentum_step(params, grads, velocities, lr, mom=0.9):
    """SGD with momentum. Velocity accumulates past gradients."""
    for p, g, v in zip(params, grads, velocities):
        v[:] = mom * v + g
        p -= lr * v

def rmsprop_step(params, grads, sq_avgs, lr, beta=0.9, eps=1e-8):
    """RMSProp: adapt learning rate by gradient magnitude."""
    for p, g, v in zip(params, grads, sq_avgs):
        v[:] = beta * v + (1 - beta) * g**2
        p -= lr * g / (np.sqrt(v) + eps)

## Dampened vs Classical Momentum

In [ ]:
# Classical momentum (fast.ai/PyTorch SGD default)
mom, grad = 0.9, 1.0
v_classical = 0.0
v_classical = mom * v_classical + grad
print(f"Classical momentum update: v = {v_classical}")

# Dampened momentum (used inside Adam)
v_dampened = 0.0
v_dampened = mom * v_dampened + (1 - mom) * grad
print(f"Dampened momentum update:  v = {v_dampened}")

## Exponential Moving Average and Debiasing

In [ ]:
def ema_with_debiasing(values, beta=0.9):
    """Exponential moving average with bias correction."""
    avg = 0.0
    corrected = []
    uncorrected = []
    for t, x in enumerate(values, 1):
        avg = beta * avg + (1 - beta) * x
        uncorrected.append(avg)
        corrected.append(avg / (1 - beta**t))
    return uncorrected, corrected

# Generate noisy signal around 10
np.random.seed(42)
signal = 10 + np.random.randn(100)

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
betas = [0.5, 0.7, 0.9, 0.99]

for ax, beta in zip(axes, betas):
    uncorrected, corrected = ema_with_debiasing(signal, beta)
    ax.plot(signal, alpha=0.3, color='gray', linewidth=0.5, label='Signal')
    ax.plot(uncorrected, color='#ef4444', linewidth=1.5, label='EMA (raw)')
    ax.plot(corrected, color='#3b82f6', linewidth=1.5, label='EMA (debiased)')
    ax.set_title(f'β = {beta}')
    ax.set_xlabel('Step')
    if ax == axes[0]:
        ax.set_ylabel('Value')
        ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('EMA Debiasing at Different β Values', fontsize=14)
plt.tight_layout()
plt.show()

## Adam: Full Implementation

Adam combines momentum and RMSProp with bias correction.

In [ ]:
class AdamNumPy:
    """Adam optimiser from scratch."""

    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = {}  # first moments
        self.v = {}  # second moments
        self.t = 0

    def step(self, params, grads):
        self.t += 1
        for i, (p, g) in enumerate(zip(params, grads)):
            if i not in self.m:
                self.m[i] = np.zeros_like(p)
                self.v[i] = np.zeros_like(p)

            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g**2

            m_hat = self.m[i] / (1 - self.beta1**self.t)
            v_hat = self.v[i] / (1 - self.beta2**self.t)

            p -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

## Verifying Against PyTorch

In [ ]:
import torch

# Shared setup
np.random.seed(42)
w_np = np.array([1.0, -2.0, 0.5])
g_np = np.array([0.1, -0.3, 0.2])

# NumPy Adam
adam_np = AdamNumPy(lr=0.001)
w_test = w_np.copy()
adam_np.step([w_test], [g_np.copy()])

# PyTorch Adam
w_pt = torch.tensor(w_np, requires_grad=True)
opt = torch.optim.Adam([w_pt], lr=0.001)
w_pt.grad = torch.tensor(g_np)
opt.step()

print(f"NumPy:   {w_test}")
print(f"PyTorch: {w_pt.detach().numpy()}")
print(f"Match:   {np.allclose(w_test, w_pt.detach().numpy(), atol=1e-7)}")

## Weight Decay vs L2 Regularisation

L2 regularisation modifies the gradient; decoupled weight decay modifies the parameter directly. For Adam, these are **not** equivalent.

In [ ]:
# Demonstrate the difference
param = np.array([2.0, -1.5, 0.8])
grad = np.array([0.1, -0.2, 0.15])
lr, wd = 0.01, 0.1

# L2 regularisation: modifies the gradient
param_l2 = param.copy()
grad_l2 = grad + wd * param_l2
param_l2 -= lr * grad_l2
print(f"L2 regularisation: {param_l2}")

# Decoupled weight decay: modifies the parameter directly
param_wd = param.copy()
param_wd *= (1 - lr * wd)
param_wd -= lr * grad
print(f"Weight decay:      {param_wd}")
print(f"Same result:       {np.allclose(param_l2, param_wd)}")

## Training a CNN: SGD vs Adam in Practice

Benchmark on MNIST with the SimpleCNN architecture.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_data = datasets.MNIST('data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('data', train=False, transform=transform)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1000)

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=5, padding=2, stride=2)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1, stride=2)
        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, padding=1, stride=2)
        self.fc = nn.Linear(32 * 4 * 4, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = x.view(x.size(0), -1)
        return self.fc(x)

def train_and_evaluate(opt_class, opt_kwargs, epochs=5, seed=42):
    torch.manual_seed(seed)
    model = SimpleCNN()
    optimiser = opt_class(model.parameters(), **opt_kwargs)
    results = []

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            loss = F.cross_entropy(model(images), labels)
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()

        model.eval()
        correct = 0
        with torch.no_grad():
            for images, labels in test_loader:
                correct += (model(images).argmax(1) == labels).sum().item()
        results.append(correct / len(test_data) * 100)
    return results

In [ ]:
configs = {
    "SGD (lr=0.01)": (torch.optim.SGD, {"lr": 0.01}),
    "SGD+Momentum (lr=0.01)": (torch.optim.SGD, {"lr": 0.01, "momentum": 0.9}),
    "Adam (lr=0.001)": (torch.optim.Adam, {"lr": 0.001}),
    "AdamW (lr=0.001)": (torch.optim.AdamW, {"lr": 0.001, "weight_decay": 0.01}),
}

all_results = {}
for name, (opt_cls, kwargs) in configs.items():
    acc = train_and_evaluate(opt_cls, kwargs)
    all_results[name] = acc
    print(f"{name:25s} -> {[f'{a:.1f}%' for a in acc]}")

In [ ]:
# Plot MNIST training curves
fig, ax = plt.subplots(figsize=(10, 6))
colors = {'SGD (lr=0.01)': '#ef4444', 'SGD+Momentum (lr=0.01)': '#3b82f6',
          'Adam (lr=0.001)': '#f59e0b', 'AdamW (lr=0.001)': '#22c55e'}
epochs = range(1, 6)

for name, acc in all_results.items():
    ax.plot(epochs, acc, 'o-', color=colors[name], linewidth=2, markersize=8, label=name)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Optimiser Comparison on MNIST (SimpleCNN)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(85, 100)
plt.tight_layout()
plt.show()

## Exercises

1. **Nesterov momentum** — Modify `sgd_momentum` to evaluate the gradient at the look-ahead position `x - lr * mom * v` instead of the current position. Compare trajectories on Rosenbrock.

2. **Learning rate sensitivity** — Run Adam on Rosenbrock with lr=0.001, 0.01, 0.05, 0.1. At what point does it diverge?

3. **AdamW vs Adam** — Train SimpleCNN with both `torch.optim.Adam` and `torch.optim.AdamW` (weight_decay=0.01) for 10 epochs. Plot both curves. Does AdamW generalise better?

4. **Bias correction ablation** — Remove bias correction from the Adam implementation (use `m` and `v` directly instead of `m_hat` and `v_hat`). Run on Rosenbrock and compare the first 50 steps.

5. **LAMB optimiser** — Implement LAMB by scaling Adam's update by `||params|| / ||update||` for each layer. Does it allow a larger batch size on MNIST?

## References

- Kingma, D.P. & Ba, J. (2014). [Adam: A Method for Stochastic Optimization.](https://arxiv.org/abs/1412.6980)
- Loshchilov, I. & Hutter, F. (2019). [Decoupled Weight Decay Regularization.](https://arxiv.org/abs/1711.05101)
- Robbins, H. & Monro, S. (1951). A Stochastic Approximation Method. Annals of Mathematical Statistics.
- Polyak, B.T. (1964). Some methods of speeding up the convergence of iteration methods. USSR Computational Mathematics and Mathematical Physics.
- fast.ai course: [Practical Deep Learning for Coders, Lesson 11](https://course.fast.ai/).